# conv-windowing-2d — ex2: extend the conv2d window view to stride S

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-windowing-2d`. Running the final beacon cell reports progress against the `CNN: 2-D conv windowing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 2-D conv windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-windowing-2d`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-windowing-2d"
DD_SUBTOPIC = "CNN: 2-D conv windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Strided 2-D windowing via `as_strided` — quick refresher

Generalizing the stride-1 case: given `x: (B, IC, H, W)`, kernel `(KH, KW)`, and conv stride `(SH, SW)`, the strided window view is:

```
x.as_strided(
    size=(B, IC, OH, OW, KH, KW),
    stride=(s_b, s_ic, s_h * SH, s_w * SW, s_h, s_w),
)
```

where `OH = (H - KH) // SH + 1` and `OW = (W - KW) // SW + 1`.

**The only change from stride-1.** The MIDDLE pair `(s_h, s_w)` — which walks *between* windows — gets multiplied by `(SH, SW)`. The trailing pair `(s_h, s_w)` — which walks *within* a window — is unchanged (we always read every pixel inside a window).

**Why.** Adjacent windows along the new `OH` axis used to be 1 input-row apart (stride 1). Now they're `SH` input-rows apart, so each step in `OH` moves `s_h * SH` storage positions. Same for width.

**Equivalence.** Contract via `einops.einsum(..., 'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')` and the result equals `F.conv2d(x, weight, stride=(SH, SW))` to fp tolerance.

### Exercise 2 — extend the conv2d window view to stride S

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `as_strided` with stride-multiplied between-window advances to build a `(B, IC, OH, OW, KH, KW)` window view of a 2-D input for arbitrary conv stride `(SH, SW)`, and verify einsum-with-kernel matches `F.conv2d(stride=...)`.
> Keywords: as_strided, windowing-2d, stride, conv2d-strided
> ```

**KCs targeted:** `windowing-stride-multiplier`, `strided-conv-output-shape`

Implement `ex2_conv2d_windows_strided(x, KH, KW, SH, SW)`. Given `x: (B, IC, H, W)`, kernel sizes `(KH, KW)`, and conv strides `(SH, SW)`, return the strided window view of shape `(B, IC, OH, OW, KH, KW)` where:

- `OH = (H - KH) // SH + 1`
- `OW = (W - KW) // SW + 1`
- Each `(KH, KW)` slice along `(OH, OW)` is one kernel-sized window of `x` at conv stride `(SH, SW)`.

**The trick (generalized from stride-1).** Read `x.stride()` = `(s_b, s_ic, s_h, s_w)`, then:

```
x.as_strided(
    size=(B, IC, OH, OW, KH, KW),
    stride=(s_b, s_ic, s_h * SH, s_w * SW, s_h, s_w),
)
```

**The only change from stride-1.** The MIDDLE pair `(s_h * SH, s_w * SW)` — between-window step — gets multiplied by the conv stride. The TRAILING pair `(s_h, s_w)` — within-window step — is unchanged.

**Why.** Adjacent windows in `OH` are now `SH` input rows apart (not 1). So moving 1 along `OH` moves `SH` rows in storage, i.e., `s_h * SH` elements. The within-window axes always read every position, so they keep `s_h` and `s_w`.

**Constraints.** No copy — return must share storage with `x`.

The test contracts against a random kernel via `einops.einsum` and compares to `F.conv2d(x, weight, stride=(SH, SW))` for multiple `(SH, SW)` combos.

In [ ]:
def ex2_conv2d_windows_strided(x: Tensor, KH: int, KW: int, SH: int, SW: int) -> Tensor:
    B, IC, H, W = x.shape
    OH = (H - KH) // SH + 1
    OW = (W - KW) // SW + 1
    s_b, s_ic, s_h, s_w = x.stride()
    return x.as_strided(
        size=(B, IC, OH, OW, KH, KW),
        stride=(s_b, s_ic, s_h * SH, s_w * SW, s_h, s_w),
    )


<details><summary>Solution</summary>

```python
def ex2_conv2d_windows_strided(x: Tensor, KH: int, KW: int, SH: int, SW: int) -> Tensor:
    B, IC, H, W = x.shape
    OH = (H - KH) // SH + 1
    OW = (W - KW) // SW + 1
    s_b, s_ic, s_h, s_w = x.stride()
    return x.as_strided(
        size=(B, IC, OH, OW, KH, KW),
        stride=(s_b, s_ic, s_h * SH, s_w * SW, s_h, s_w),
    )
```

**The single load-bearing change.** Compare to the stride-1 version: only the middle pair `(s_h, s_w)` becomes `(s_h * SH, s_w * SW)`. Everything else is identical. The trailing pair stays `(s_h, s_w)` because we always step ONE row down within a window, regardless of conv stride.

**Why two pairs of `(s_h, s_w)` exist at all.** The window view introduces TWO new axes for height (`OH` for between-window, `KH` for within-window) and TWO for width. Each new axis needs its own stride. The same input row participates in both 'I am the start of window N' and 'I am the M-th row of window N-1' — so the stride values overlap but the semantic roles don't.

**Connecting to the output shape.** `OH = (H - KH) // SH + 1` is the same formula from `conv-stride-downsample`. Window count = valid kernel positions along the axis. The leading window starts at index 0; subsequent windows step by `SH`; the last must fit entirely (floor division).

**For padding too.** Compose with the `conv-padding-zero` drill: pre-pad `x` by `F.pad`, then window the padded tensor with this strided view. That's the full ARENA conv2d implementation in two composable atoms.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()